# 長江杯 2026 — A100 Colab 自動跑批與自癒斷點續跑 Worker（Ultra 5x 1,000 CU 省額度版）

**核心防中斷與自癒特性**：
- 🔍 **第 1 格自動診斷 (`--diagnose`)**：一鍵檢查 Google Drive 上每個階段（`s1_mono`, `s23`, `s1_trimer`, `s23_var`）已完成的 PDB 數量與 `worker.log` 最後 12 行記錄。
- 🔄 **第 4 格自動找回未完任務 (`[AUTO-RECOVER]`)**：即使上次異常斷線把任務單移進了 `jobs/completed/` 或 `jobs/failed/`，只要檢測到有任一階段未滿 100%，第 4 格會**自動把任務單拉回 `jobs/running/` 並從斷點繼續跑**（已算完的序列 0 秒跳過）！
- 🛡️ **智慧釋放 GPU（15 秒倒數）**：僅當所有階段 100% 驗證完成後才倒數 15 秒執行 `runtime.unassign()`。
- ⚡ **第 5 格黃金凹槽骨架 v2.1 採收 + 一鍵下載 AlphaFold 3 JSON**：針對多鏈凹槽靶點（如 Target 5 TL1A），可單獨運行第 5 格（免裝 BindCraft、便宜 T4/L4 GPU 僅需 ~1–2 分鐘），自動對 `Trajectory/Relaxed/` 凹槽骨架跑 **三模式 SolubleMPNN (`core_fix` + `full` + 受控 `fix_intf`, 禁用 `CXM`) + ESMFold 單體 `scRMSD` 自洽門控 + `3K51` DcR3 受體競爭重疊 + 3D 姿態聚類**，並自動將更新後的 **`1×Binder + 3×野生型 TL1A` AlphaFold 3 Server JSON** 下載到電腦！

In [ ]:
#@title 1. 檢查 GPU 架構、掛載共享 Google Drive (1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg)、同步 GitHub 並印出診斷報告
import os, sys, subprocess, pathlib, importlib, torch
from google.colab import drive

SHARED_DRIVE_URL = "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link" #@param {type:"string"}
GITHUB_REPO_URL = "https://github.com/David-Liv/changjiang-protein-2026.git" #@param {type:"string"}
REPO_DIR = "/content/changjiang-protein-2026"

print("=== [1/4] GPU 架構與 1,000 CU 額度自檢 ===")
!nvidia-smi
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1e9
    sm = f"sm_{p.major}{p.minor}"
    print(f"檢測到 GPU: {p.name} | 顯存: {gb:.1f} GB | 架構: {sm}")
    if p.major >= 12 or "BLACKWELL" in p.name.upper() or "RTX PRO 6000" in p.name.upper():
        print("❌ [警告] 當前為 G4 (Blackwell sm_120)！BindCraft (CUDA 12.4/JAX) 不支持此架構，請切換為 A100！")
    elif "H100" in p.name.upper() or p.major == 9:
        print("⚠️ [提示] 當前為 H100 (~18 CU/h，1,000 CU 約可跑 55 小時)。日常跑批建議改選 A100 以節省額度。")
    else:
        print("✅ [最佳配置] A100 架構完全兼容 ESMFold 與 BindCraft（~5.4–7.5 CU/h，1,000 CU 可跑 135–185 小時）✓")
else:
    print("❌ 未檢測到 GPU！請在『執行階段 -> 變更執行階段類型』選擇 A100 GPU。")

print("\n=== [2/4] 檢查 ESMFold / ipSAE 評估依賴（不升級 numpy，防止內核崩潰）===")
!pip install -q transformers accelerate

print("\n=== [3/4] 掛載 Google Drive（同一執行階段只驗證 1 次，永不重複卸載）===")
if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✅ Google Drive 已處於掛載狀態，免重複驗證！")

print("\n=== [4/4] 同步 GitHub 最新代碼並直連團隊共享資料夾 (ID: 1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg) ===")
if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main && git -C {REPO_DIR} reset --hard origin/main

%cd {REPO_DIR}
import importlib.util
sys.path_importer_cache.clear()
importlib.invalidate_caches()
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"), ("worker", f"{REPO_DIR}/pipeline/worker.py")]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
worker = sys.modules["worker"]

# 嚴格鎖定共享資料夾 ID，絕不在私人 MyDrive 自動 mkdir 新資料夾！
DRIVE_WORKSPACE = str(worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL))
print(f"🔗 已鎖定團隊共享網盤實際路徑：{DRIVE_WORKSPACE}")

!python3 scripts/tnf_pipeline.py selftest
worker.main(["--drive-root", DRIVE_WORKSPACE, "--diagnose"])

In [ ]:
#@title 2. 極速載入/安裝 BindCraft + SolubleMPNN + AF2 權重（可獨立執行：自動掛載網盤 + `.tar.zst` 鏡像快取 45 秒還原！）
import os, sys, time, pathlib, importlib, importlib.util, torch
from google.colab import drive

SAVE_ENV_CACHE_TO_DRIVE = True #@param {type:"boolean"}
SHARED_DRIVE_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
REPO_DIR = "/content/changjiang-protein-2026"

if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 12:
    raise RuntimeError("當前 GPU 為 G4 (Blackwell sm_120)，與 BindCraft CUDA 12.4 不兼容！請先切換執行階段為 A100 再安裝。")

if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    print("🔄 偵測到 A100 重連後尚未掛載 Google Drive，正在自動掛載...")
    drive.mount('/content/drive')

if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main && git -C {REPO_DIR} reset --hard origin/main

sys.path_importer_cache.clear()
importlib.invalidate_caches()
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"), ("worker", f"{REPO_DIR}/pipeline/worker.py")]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
worker = sys.modules["worker"]

shared_root = worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL)
cache_dir = shared_root / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
cache_tar = cache_dir / "bindcraft_colab_cache.tar.zst"

bc_python = "/content/miniforge3/envs/BindCraft/bin/python"
bc_params_file = "/content/BindCraft/params/params_model_1_ptm.npz"

# 自動修復：若先前在 /content 下執行了 install_bindcraft.sh，5.2GB 的 AF2 權重會落在 /content/params，0.1 秒自動歸位！
if os.path.exists("/content/params/params_model_1_ptm.npz") and not os.path.exists(bc_params_file):
    print("🔧 偵測到已下載的 5.2GB AlphaFold2 權重位於 /content/params，正在 0.1 秒自動移入 /content/BindCraft/params ...")
    !rm -rf /content/BindCraft/params && mv /content/params /content/BindCraft/params
    !chmod +x /content/BindCraft/functions/dssp /content/BindCraft/functions/DAlphaBall.gcc

if (not os.path.exists(bc_python) or not os.path.exists(bc_params_file)) and cache_tar.exists():
    t0 = time.time()
    print(f"🚀 偵測到共享網盤已存在預編譯環境快取 ({cache_tar.name})！正在多核極速解壓至本地 NVMe SSD（約 45–60 秒）...")
    !apt-get install -y -qq zstd
    !tar -I 'zstd -d -T0' -xf "{cache_tar}" -C /
    print(f"⚡ 環境快取極速還原完成，耗時 {time.time() - t0:.1f} 秒！（省去 ~12 分鐘安裝時間）")

if not os.path.exists("/content/miniforge3/bin/conda"):
    print("正在安裝 Miniforge3（隔離環境，不污染 Colab 主內核 PATH）...")
    !wget -q -O /content/Miniforge3.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash /content/Miniforge3.sh -b -p /content/miniforge3

if not os.path.exists("/content/BindCraft/bindcraft.py"):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft

if not os.path.exists(bc_python) or not os.path.exists(bc_params_file):
    print("正在首次安裝 BindCraft 環境與下載 AlphaFold2 權重（約需 8–12 分鐘，完成後將自動打包快取至網盤，以後只需 45 秒！）...")
    !cd /content/BindCraft && PATH="/content/miniforge3/bin:$PATH" bash install_bindcraft.sh --cuda '12.4' --pkg_manager 'conda'

if os.path.exists(bc_python) and os.path.exists(bc_params_file):
    !chmod +x /content/BindCraft/functions/dssp /content/BindCraft/functions/DAlphaBall.gcc
    print("✅ BindCraft + SolubleMPNN + PyRosetta + AlphaFold2 權重已完整就緒 ✓")
    if SAVE_ENV_CACHE_TO_DRIVE and not cache_tar.exists():
        t1 = time.time()
        print(f"📦 正在將安裝好的環境清理並多核壓縮存入共享網盤 ({cache_tar})，供你與全隊今後 45 秒秒開...")
        !/content/miniforge3/bin/conda clean -a -y > /dev/null 2>&1 || true
        !apt-get install -y -qq zstd
        !tar -I 'zstd -1 -T0' -cf "/content/bindcraft_colab_cache.tar.zst.tmp" /content/miniforge3 /content/BindCraft
        !mv -f "/content/bindcraft_colab_cache.tar.zst.tmp" "{cache_tar}"
        print(f"🎉 網盤環境快取打包完成（耗時 {time.time() - t1:.1f} 秒）！下次開新 A100 只需 45 秒即可直接開跑！")
else:
    raise RuntimeError("❌ BindCraft 安裝未完成！請檢查上方日誌是否有網絡或安裝報錯。")

In [ ]:
#@title 3. 將預設任務模板投遞到共享網盤 `jobs/pending/`（支援多輪自動遞增批次號，隨點隨跑！）
import os, re, sys, json, shutil, pathlib, importlib.util
from google.colab import drive

# 說明：MAX_TRAJECTORIES 設為 0 時沿用任務模板內建軌跡數（003 互補批次預設 22 條 ≈ 45 CU；002 主力批次預設 35 條）；大於 0 時覆寫。
TEMPLATE_CHOICE = "target5_TL1A_grooveAB_003_B317_mixed" #@param ["target5_TL1A_grooveAB_003_B317_mixed", "target5_TL1A_minibinder", "target1_EGFR_vhh", "target2_TNFa_calibration", "none"]
MAX_TRAJECTORIES = 0 #@param {type:"integer"}
SHARED_DRIVE_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
REPO_DIR = "/content/changjiang-protein-2026"
if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main && git -C {REPO_DIR} reset --hard origin/main

for _mname, _mpath in [("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"), ("worker", f"{REPO_DIR}/pipeline/worker.py")]:
    if _mname not in sys.modules:
        _spec = importlib.util.spec_from_file_location(_mname, _mpath)
        _mod = importlib.util.module_from_spec(_spec)
        sys.modules[_mname] = _mod
        _spec.loader.exec_module(_mod)
worker = sys.modules["worker"]

shared_root = worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL)
pending_dir = shared_root / "jobs" / "pending"
running_dir = shared_root / "jobs" / "running"
runs_dir = shared_root / "runs"
pending_dir.mkdir(parents=True, exist_ok=True)
running_dir.mkdir(parents=True, exist_ok=True)

mapping = {
    "target5_TL1A_grooveAB_003_B317_mixed": "/content/changjiang-protein-2026/jobs/templates/job_target5_TL1A_grooveAB_003.json",
    "target5_TL1A_minibinder": "/content/changjiang-protein-2026/jobs/templates/job_target5_TL1A_minibinder.json",
    "target1_EGFR_vhh": "/content/changjiang-protein-2026/jobs/templates/job_target1_EGFR_vhh.json",
    "target2_TNFa_calibration": "/content/changjiang-protein-2026/jobs/templates/job_target2_TNFa_calibration.json",
}
if TEMPLATE_CHOICE in mapping:
    src = pathlib.Path(mapping[TEMPLATE_CHOICE])
    if (running_dir / src.name).exists():
        print(f"🔄 任務 {src.name} 已在共享網盤 jobs/running/ 中，直接運行『第 4 格』即可從斷點繼續！")
    else:
        jdata = json.loads(src.read_text(encoding="utf-8"))
        if "max_trajectories" in jdata and int(MAX_TRAJECTORIES) > 0:
            jdata["max_trajectories"] = int(MAX_TRAJECTORIES)
        # 若上一批同名前綴任務已 COMPLETED，自動遞增批次號 (_002 -> _003 -> _004) 以便連續慢跑累積新序列！
        jid = str(jdata.get("job_id", src.stem))
        m = re.match(r"^(.*_)(\d{3,})$", jid)
        if m and jdata.get("stage", "generation") == "generation":
            prefix, num_s = m.group(1), m.group(2)
            num = int(num_s)
            while True:
                cand_id = f"{prefix}{num:0{len(num_s)}d}"
                mf = runs_dir / cand_id / "run_manifest.json"
                if mf.exists():
                    try:
                        if json.loads(mf.read_text(encoding="utf-8")).get("status") == "COMPLETED":
                            num += 1
                            continue
                    except Exception:
                        pass
                jid = cand_id
                break
            jdata["job_id"] = jid
        dst = pending_dir / src.name
        dst.write_text(json.dumps(jdata, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"✅ 已投遞任務單 (job_id={jdata.get('job_id')}, max_trajectories={jdata.get('max_trajectories', '-')}) 至共享網盤：{dst}")

In [ ]:
#@title 4. 啟動隊列執行器（顯存常駐 + 同長度 Batch 並行加速 + 永不卸載網盤免重複驗證）
import os, sys, time, importlib, importlib.util
from google.colab import runtime

# 提示：若你要連續跑多個任務，請取消勾選 AUTO_UNASSIGN_GPU，這樣不會斷開 A100 也不需重新驗證網盤！
# 若你要掛機睡覺/離開，請勾選 AUTO_UNASSIGN_GPU，跑完 100% 後會自動釋放 A100 省額度。
AUTO_UNASSIGN_GPU = True #@param {type:"boolean"}
REPO_DIR = "/content/changjiang-protein-2026"
if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main && git -C {REPO_DIR} reset --hard origin/main

sys.path_importer_cache.clear()
importlib.invalidate_caches()
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 直接透過絕對檔案路徑載入模組（100% 免疫 ModuleNotFoundError，且消除編輯器黃色波浪線警告）
for _mname, _mpath in [("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"), ("worker", f"{REPO_DIR}/pipeline/worker.py")]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
worker = sys.modules["worker"]

SHARED_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
DRIVE_WORKSPACE = str(worker.resolve_shared_drive_workspace(SHARED_URL))
argv = ["--drive-root", DRIVE_WORKSPACE, "--once"]
if AUTO_UNASSIGN_GPU:
    argv.append("--unassign-on-exit")

# 在 Colab 主 IPython Kernel 內直接執行（不呼叫 drive.flush_and_unmount()，避免下次要重新驗證 Google Drive！）
exit_code = worker.main(argv)

if AUTO_UNASSIGN_GPU and exit_code == 0:
    if hasattr(os, "sync"):
        os.sync()
    try:
        from google.colab import drive as _colab_drive
        _colab_drive.flush_and_unmount()
    except Exception:
        pass
    print("🔌 正在從 Colab 主內核強制釋放 A100 GPU (runtime.unassign)...", flush=True)
    time.sleep(2)
    runtime.unassign()

In [ ]:
#@title 5. ⚡ 黃金凹槽骨架 v2.1 極速採收（自動網盤快取 ESMFold/SolubleMPNN 權重 + 序列級結果快取 + 3K51 競爭/姿態聚類）
# 說明：首次執行會自動將 ESMFold (~8.5GB) 與 SolubleMPNN 權重打包存入共享網盤 cache/（以後新開 T4/A100 僅 15 秒秒開免下載！），並自動快取已摺疊單體結果。
import os, sys, time, pathlib, importlib, importlib.util
from google.colab import drive, files, runtime

JOB_ID = "job_t5_tl1a_grooveAB_002" #@param {type:"string"}
MIN_TRAJ_IPTM = 0.75 #@param {type:"number"}
NUM_SEQS_CORE_FIX = 6 #@param {type:"integer"}
NUM_SEQS_FULL_REDESIGN = 6 #@param {type:"integer"}
NUM_SEQS_FIX_INTERFACE = 4 #@param {type:"integer"}
OMIT_AAS = "CXM" #@param {type:"string"}
MAX_KEEP_PER_BACKBONE = 2 #@param {type:"integer"}
RUN_ESMFOLD_SCRMSD_GATE = True #@param {type:"boolean"}
AUTO_DOWNLOAD_AF3_JSON = True #@param {type:"boolean"}
AUTO_UNASSIGN_GPU_AFTER = False #@param {type:"boolean"}

SHARED_DRIVE_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
REPO_DIR = "/content/changjiang-protein-2026"

if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    print("🔄 正在掛載 Google Drive...")
    drive.mount('/content/drive')

if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main && git -C {REPO_DIR} reset --hard origin/main

sys.path_importer_cache.clear()
importlib.invalidate_caches()
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)

for _mname, _mpath in [
    ("check_novelty", f"{REPO_DIR}/scripts/check_novelty.py"),
    ("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"),
    ("worker", f"{REPO_DIR}/pipeline/worker.py"),
    ("harvest_groove_backbones", f"{REPO_DIR}/scripts/harvest_groove_backbones.py"),
]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)

worker = sys.modules["worker"]
hgb = sys.modules["harvest_groove_backbones"]
drive_root = worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL)

cli_args = [
    "--drive-root", str(drive_root),
    "--job-id", JOB_ID.strip(),
    "--min-iptm", str(MIN_TRAJ_IPTM),
    "--num-seqs-core-fix", str(NUM_SEQS_CORE_FIX),
    "--num-seqs-full", str(NUM_SEQS_FULL_REDESIGN),
    "--num-seqs-fix-intf", str(NUM_SEQS_FIX_INTERFACE),
    "--omit-aas", OMIT_AAS.strip() or "CXM",
    "--max-per-backbone", str(MAX_KEEP_PER_BACKBONE),
]
if not RUN_ESMFOLD_SCRMSD_GATE:
    cli_args.append("--skip-esmfold")

hgb.main(cli_args)

af3_dir = drive_root / "runs" / JOB_ID.strip() / "af3_inputs"
af3_jsons = sorted(af3_dir.glob(f"af3_server_{JOB_ID.strip()}_groove*.json"))
if AUTO_DOWNLOAD_AF3_JSON and af3_jsons:
    print(f"\n📥 正在自動觸發瀏覽器下載 {len(af3_jsons)} 個 AlphaFold 3 Server 批量 JSON 檔案（若瀏覽器攔截多檔下載，請允許彈窗，或直接從雲盤 af3_inputs/ 取用）：")
    for jp in af3_jsons:
        print(f"   ⬇️ {jp.name}")
        files.download(str(jp))
        time.sleep(1.5)

if AUTO_UNASSIGN_GPU_AFTER:
    if hasattr(os, "sync"):
        os.sync()
    print("⏳ 5 秒後自動釋放 GPU 連線...")
    time.sleep(5)
    runtime.unassign()